In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/02_fact_observations
# Purpose  : Largest Fact table (~16.2M rows). Grain: One row per test/measurement.
#
# Key Refinements & Architectural Decisions:
#   1) Decoupled observation attributes into dim_observation_type to eliminate repetition.
#   2) Re-introduced observation_key (PK) for Unity Catalog Informational Constraints
#      and BI metadata optimization (consistent with all Gold Fact tables).
#   3) Liquid Clustering applied on ["patient_key", "date_key"] to optimize range queries
#      and time-series value drift analysis.
#   4) Kept encounter_id as a Degenerate Dimension (nullable) and preserved full
#      observation_datetime precision for window functions (LAG/LEAD).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_observations",
    comment="Fact table - one row per clinical observation",
    cluster_by=["patient_key", "date_key"],
)
def fact_observations():
    observations = dlt.read_stream("covid19_lakehouse.silver.observations")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    observation_types = spark.read.table("covid19_lakehouse.gold.dim_observation_type").select(
        F.col("observation_code"), F.col("observation_type_key")
    )

    df = (
        observations
        .join(patients, on="patient_id", how="left")
        .join(observation_types, on="observation_code", how="left")
    )

    return df.select(
        F.xxhash64(
            "patient_id", "encounter_id", "observation_code", "observation_datetime"
        ).alias("observation_key"),
        F.col("patient_key"),
        F.col("encounter_id"),  # Degenerate Dimension
        F.col("observation_type_key"),
        F.date_format(F.col("observation_datetime"), "yyyyMMdd").cast("int").alias("date_key"),
        F.col("observation_datetime"),
        F.col("value_numeric"),
        F.col("value_text"),
    )